# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

##Nombre: José Armando Sarango Cuenca

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [4]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'wikipedia-text-corpus-for-nlp-and-llm-projects' dataset.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [5]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [6]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [8]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [9]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [10]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [11]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.1 MB/s eta 0:00:00


In [12]:
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

D, I = index.search(query_vec, k=10)
print(f"¿El índice está entrenado?: {index.is_trained}")
print(f"Total de vectores indexados en FAISS: {index.ntotal}")

¿El índice está entrenado?: True
Total de vectores indexados en FAISS: 79104


In [14]:
# 4. Configurar la consulta y realizar la búsqueda
k_busqueda = 10

# Ejecutar la búsqueda a partir de tu variable query_vec
Distancias, Indices = index.search(query_vec, k=k_busqueda)

print(f"--- TOP {k_busqueda} RESULTADOS RECUPERADOS CON FAISS ---")

# Iterar sobre los resultados devueltos (vienen en matrices de dos dimensiones [0])
for i, (idx, dist) in enumerate(zip(Indices[0], Distancias[0])):
    if idx != -1:
        row_asociada = chunks_df.iloc[idx]

        print(f"Resultado #{i+1}")
        print(f" -> Índice de Fila (ID): {idx}")
        print(f" -> Distancia L2: {dist:.4f}")
        print(f" -> Metadata [Doc ID: {row_asociada['doc_id']}, Chunk ID: {row_asociada['chunk_id']}]")
        print(f" -> Fragmento de Texto: {row_asociada['text'][:150]}...")
        print("-" * 50)


--- TOP 10 RESULTADOS RECUPERADOS CON FAISS ---
Resultado #1
 -> Índice de Fila (ID): 10176
 -> Distancia L2: 0.2593
 -> Metadata [Doc ID: 1391, Chunk ID: 0]
 -> Fragmento de Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------------------------------------
Resultado #2
 -> Índice de Fila (ID): 1
 -> Distancia L2: 0.2764
 -> Metadata [Doc ID: 1, Chunk ID: 0]
 -> Fragmento de Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------------------------------------
Resultado #3
 -> Índice de Fila (ID): 10177
 -> Distancia L2: 0.3198
 -> Metadata [Doc ID: 1391, Chunk ID: 1]
 -> Fragmento de Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [15]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 11.8 MB/s eta 0:00:00


In [16]:
# Levantar  con una instancia de Qdrant.

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. Conectar con una instancia de Qdrant en memoria por que se corre e forma local
qdrant_client = QdrantClient(":memory:")

D = embeddings.shape[1]
COLLECTION_NAME = "wikipedia_collection"

print(f"Dimensión de los embeddings: {D}")


Dimensión de los embeddings: 768


In [17]:
#crear la colección
qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=D,
        distance=Distance.COSINE
    ),
)
print(f"Colección '{COLLECTION_NAME}' creada exitosamente.")


Colección 'wikipedia_collection' creada exitosamente.


/tmp/ipykernel_798/2353985830.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


In [18]:
##3Insertar los vectores en la metedata
points = []

for idx, row in chunks_df.iterrows():
    point_id = int(idx)
    vector = embeddings[idx].tolist()

    # Creamos el payload con la metadata requerida
    payload = {
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"]),
        "text": row["text"]
    }

    points.append(PointStruct(id=point_id, vector=vector, payload=payload))

# Insertar todos los puntos en la colección
qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)
print(f"Se insertaron {len(points)} puntos en Qdrant.")


Se insertaron 79104 puntos en Qdrant.


/tmp/ipykernel_798/182921137.py:18: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 79104 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client.upsert(


In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def qdrant_search(query_embedding, k=5):
    global qdrant_client, COLLECTION_NAME

    if hasattr(query_embedding, "ndim") and query_embedding.ndim > 1:
        vector_busqueda = query_embedding.flatten().tolist()
    else:
        vector_busqueda = list(query_embedding)

    search_result = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector_busqueda,
        limit=k,
        with_payload=True,
        with_vectors=False
    )

    results = []
    for hit in search_result.points:   # <-- fix: use .points
        text = hit.payload.get("text", "")
        metadata = {k: v for k, v in hit.payload.items() if k != "text"}
        results.append((hit.id, hit.score, text, metadata))

    return results

# ejemplo de una consulta pra k=5
k_ejemplo = 5
resultados_qdrant = qdrant_search(query_vec, k=k_ejemplo)

print(f"--- TOP {k_ejemplo} RESULTADOS EN QDRANT ---")
for res in resultados_qdrant:
    idx, score, text, meta = res
    print(f"ID: {idx} | Score (Similitud Coseno): {score:.4f}")
    print(f"Metadata: {meta}")
    print(f"Texto: {text[:150]}...")
    print("-" * 50)


--- TOP 5 RESULTADOS EN QDRANT ---
ID: 10176 | Score (Similitud Coseno): 0.8703
Metadata: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------------------------------------
ID: 1 | Score (Similitud Coseno): 0.8618
Metadata: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------------------------------------
ID: 10177 | Score (Similitud Coseno): 0.8401
Metadata: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------------------------------------
ID: 37406 | Score (Similitud Coseno): 0.8391
Metadata: {'doc_id': 5067, 'ch

## ¿La métrica usada fue Cosine o L2? ¿Por qué?

**Respuesta:**

Usé Cosine, porque el modelo `intfloat/e5-base-v2` está entrenado para comparar textos con similitud coseno. Además generé los embeddings con `normalize_embeddings=True`, y con vectores normalizados L2 y coseno dan el mismo ranking, así que en la práctica daría igual — pero coseno es el estándar para retrieval con este tipo de modelos y hace los scores más fáciles de interpretar.


## ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?

**Respuesta:**

Mucho más fácil. FAISS solo guarda vectores, así que la metadata la tuve que manejar yo por fuera: buscar el índice en FAISS y luego ir a `chunks_df.iloc[idx]` para recuperar el texto y los IDs. En Qdrant cada punto lleva su payload (un diccionario con text, doc_id y chunk_id), así que la búsqueda ya devuelve todo junto, y si quisiera filtrar por doc_id lo haría directo en la query con su API de

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [20]:
!pip install -q "pymilvus[milvus_lite]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 33.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [21]:
#conexxion y defincion de la coleccion
import time
from pymilvus import MilvusClient, DataType

# 1. Conectar a Milvus en modo Lite
milvus_client = MilvusClient("milvus_demo.db")

COLLECTION_MILVUS = "wikipedia_milvus"
D = embeddings.shape[1]  # 768 para E5-base-v2

# Si la colección ya existe por pruebas previas, la borramos
if milvus_client.has_collection(collection_name=COLLECTION_MILVUS):
    milvus_client.drop_collection(collection_name=COLLECTION_MILVUS)

# 2. Crear el esquema estructurado
schema = milvus_client.create_schema(auto_id=False, enable_dynamic_field=False)

# Añadir los campos obligatorios y de metadata requeridos
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=D)
schema.add_field(field_name="doc_id", datatype=DataType.INT64)
schema.add_field(field_name="chunk_id", datatype=DataType.INT64)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=2048)

print("Esquema de Milvus creado exitosamente.")


Esquema de Milvus creado exitosamente.


In [26]:
# Crear Colección e Insertar Datos
import gc

if milvus_client.has_collection(collection_name=COLLECTION_MILVUS):
    milvus_client.drop_collection(collection_name=COLLECTION_MILVUS)

milvus_client.create_collection(
    collection_name=COLLECTION_MILVUS,
    schema=schema
)

# Extraer columnas UNA vez
texts = chunks_df["text"].tolist()
doc_ids = chunks_df["doc_id"].to_numpy()
chunk_ids = chunks_df["chunk_id"].to_numpy()

BATCH_SIZE = 1000
total_insertados = 0
N = len(chunks_df)

for start in range(0, N, BATCH_SIZE):
    end = min(start + BATCH_SIZE, N)

    # El lote se construye aquí y se destruye en la siguiente iteración
    batch = [
        {
            "id": int(i),
            "embedding": embeddings[i].tolist(),
            "doc_id": int(doc_ids[i]),
            "chunk_id": int(chunk_ids[i]),
            "text": texts[i],
        }
        for i in range(start, end)
    ]

    insert_result = milvus_client.insert(
        collection_name=COLLECTION_MILVUS,
        data=batch
    )
    total_insertados += insert_result["insert_count"]
    print(f"Lote {start // BATCH_SIZE + 1}: insertados {insert_result['insert_count']} (acumulado: {total_insertados})")

# Liberar lo que queda
del batch
gc.collect()

print(f"\nTotal insertados en Milvus: {total_insertados} registros.")

Lote 1: insertados 1000 (acumulado: 1000)
Lote 2: insertados 1000 (acumulado: 2000)
Lote 3: insertados 1000 (acumulado: 3000)
Lote 4: insertados 1000 (acumulado: 4000)
Lote 5: insertados 1000 (acumulado: 5000)
Lote 6: insertados 1000 (acumulado: 6000)
Lote 7: insertados 1000 (acumulado: 7000)
Lote 8: insertados 1000 (acumulado: 8000)
Lote 9: insertados 1000 (acumulado: 9000)
Lote 10: insertados 1000 (acumulado: 10000)
Lote 11: insertados 1000 (acumulado: 11000)
Lote 12: insertados 1000 (acumulado: 12000)
Lote 13: insertados 1000 (acumulado: 13000)
Lote 14: insertados 1000 (acumulado: 14000)
Lote 15: insertados 1000 (acumulado: 15000)
Lote 16: insertados 1000 (acumulado: 16000)
Lote 17: insertados 1000 (acumulado: 17000)
Lote 18: insertados 1000 (acumulado: 18000)
Lote 19: insertados 1000 (acumulado: 19000)
Lote 20: insertados 1000 (acumulado: 20000)
Lote 21: insertados 1000 (acumulado: 21000)
Lote 22: insertados 1000 (acumulado: 22000)
Lote 23: insertados 1000 (acumulado: 23000)
Lote 2

In [27]:
#Crear el Índice ANN (HNSW)
index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    metric_type="COSINE",
    index_type="HNSW",
    params={
        "M": 16,
        "efConstruction": 200
    }
)

milvus_client.create_index(
    collection_name=COLLECTION_MILVUS,
    index_params=index_params
)

# Cargar la colección en memoria para habilitar las búsquedas
milvus_client.load_collection(collection_name=COLLECTION_MILVUS)
print("Índice ANN (HNSW) construido y colección cargada en memoria.")


ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Índice ANN (HNSW) construido y colección cargada en memoria.


In [28]:
indexes = milvus_client.list_indexes(
    collection_name=COLLECTION_MILVUS
)

print(indexes)

['embedding']


In [29]:
#Definición de la Función de Búsqueda (milvus_search)
import time

def milvus_search(query_embedding, k=5, mode="ann"):
    # Aplanar el query vector a lista si es necesario
    if hasattr(query_embedding, "ndim") and query_embedding.ndim > 1:
        vector_busqueda = query_embedding.flatten().tolist()
    else:
        vector_busqueda = list(query_embedding)

    if mode == "exact":

        search_params = {"metric_type": "COSINE", "params": {"ef": max(512, k)}}
    else:
        search_params = {"metric_type": "COSINE", "params": {"ef": max(32, k)}}

    start_time = time.time()

    search_result = milvus_client.search(
        collection_name=COLLECTION_MILVUS,
        data=[vector_busqueda],
        limit=k,
        search_params=search_params,
        output_fields=["text", "doc_id", "chunk_id"]
    )

    execution_time = time.time() - start_time

    formatted_results = []
    for hit in search_result[0]:
        entity = hit["entity"]
        metadata = {
            "doc_id": entity.get("doc_id"),
            "chunk_id": entity.get("chunk_id")
        }
        formatted_results.append({
            "id": hit["id"],
            "score": hit["distance"],
            "text": entity.get("text"),
            "metadata": metadata
        })

    return formatted_results, execution_time


for k_val in [5, 20]:
    print(f"\n================ EXPERIMENTO PARA K = {k_val} ================")

    res_exact, t_exact = milvus_search(query_vec, k=k_val, mode="exact")
    ids_exact = [r["id"] for r in res_exact]

    res_ann, t_ann = milvus_search(query_vec, k=k_val, mode="ann")
    ids_ann = [r["id"] for r in res_ann]

    overlap = len(set(ids_exact).intersection(set(ids_ann)))
    porcentaje_overlap = (overlap / k_val) * 100

    print(f"[Modo EXACTO] Tiempo: {t_exact:.6f} segundos.")
    print(f"[Modo ANN]    Tiempo: {t_ann:.6f} segundos.")
    print(f"Overlap de IDs: {overlap}/{k_val} ({porcentaje_overlap:.1f}% de coincidencia)")

    print("\nMuestra de resultados ANN (Top 3):")
    for i in range(min(3, k_val)):
        print(f" -> ID: {res_ann[i]['id']} | Score: {res_ann[i]['score']:.4f} | Texto: {res_ann[i]['text'][:90]}...")


================ EXPERIMENTO PARA K = 5 ================
[Modo EXACTO] Tiempo: 4.718379 segundos.
[Modo ANN]    Tiempo: 3.413264 segundos.
Overlap de IDs: 5/5 (100.0% de coincidencia)

Muestra de resultados ANN (Top 3):
 -> ID: 10176 | Score: 0.1297 | Texto: Battery tester A battery tester is an electronic device intended for testing the state of ...
 -> ID: 1 | Score: 0.1382 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gi...
 -> ID: 10177 | Score: 0.1599 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test f...

================ EXPERIMENTO PARA K = 20 ================
[Modo EXACTO] Tiempo: 3.428333 segundos.
[Modo ANN]    Tiempo: 4.721495 segundos.
Overlap de IDs: 20/20 (100.0% de coincidencia)

Muestra de resultados ANN (Top 3):
 -> ID: 10176 | Score: 0.1297 | Texto: Battery tester A battery tester is an electronic device intended for testing the state of ...
 -> ID: 1 | Score: 0.1382 |

## ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?

**Respuesta:**

Usé un índice HNSW con `M=16` y `efConstruction=200`, que controlan cómo se construye el grafo (más conexiones y más exploración = índice más preciso pero más lento de construir).

Para la búsqueda ajusté el parámetro `ef`: en el modo "exacto" usé `ef=512`, que obliga a explorar muchos nodos y se acerca a una búsqueda exhaustiva, y en el modo "rápido" usé `ef=32`, que recorre menos nodos y responde más rápido a costa de algo de precisión. Ese es el parámetro clave del trade-off precisión vs velocidad en HNSW.


## ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?

**Respuesta:**

Comparé el overlap de IDs entre la búsqueda exacta y la ANN para k=5 y k=20. En mi caso el overlap fue del 100% en ambos, pero esto tiene una explicación: Milvus Lite (la versión local que corre en Colab) solo soporta índice FLAT internamente, así que en realidad todas las búsquedas fueron exhaustivas sin importar el valor de `ef`.

En un servidor Milvus completo con HNSW real, un `ef` bajo puede saltarse algunos vecinos óptimos y devolver otros "suficientemente cercanos", cambiando algunas posiciones o elementos del Top-k. Esa diferencia se mediría justamente con el overlap, que es la métrica que dejé implementada en el experimento.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [30]:
!pip install -q weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.5 MB/s eta 0:00:00


In [32]:
import weaviate
from weaviate.auth import AuthApiKey
import weaviate.classes as wvc
CLUSTER_URL = "https://qxtakmx3qjyqxcpihoswxg.c0.us-east-1.aws.weaviate.cloud"
API_KEY = "WW15T213blVpU2JlK05KMF9jRDEyQzdsUlk4V3cya2xQajkwekRHdGVHckpPcFFENXhmVjk0SW13dTFNPV92MjAw"

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=CLUSTER_URL,
    auth_credentials=AuthApiKey(API_KEY),
)

print("¿Conectado?:", client.is_ready())

¿Conectado?: True


In [33]:
#Definicion del esquema
COLLECTION_WEAVIATE = "WikipediaChunk"

# Si la colección ya existe por ejecuciones previas, la eliminamos
if client.collections.exists(COLLECTION_WEAVIATE):
    client.collections.delete(COLLECTION_WEAVIATE)

# 2. Definir el esquema  usando la API  de Weaviate
client.collections.create(
    name=COLLECTION_WEAVIATE,

    vectorizer_config=None,
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="chunk_id", data_type=wvc.config.DataType.INT),
    ]
)

print(f"Colección '{COLLECTION_WEAVIATE}' con esquema de propiedades creada con éxito.")


Colección 'WikipediaChunk' con esquema de propiedades creada con éxito.


In [34]:
#Inserccionde objetos
# Obtener una referencia a la colección creada
collection = client.collections.get(COLLECTION_WEAVIATE)

# 3. Insertar objetos mediante el formateador de batches nativo
print("Insertando datos en Weaviate...")

# Verificar si chunks_df está definido antes de usarlo
if 'chunks_df' not in globals():
    raise NameError("Error: 'chunks_df' no está definido. Por favor, asegúrate de ejecutar la celda que crea 'chunks_df' (Parte 1: Generación de Embeddings) antes de intentar insertar datos en Weaviate.")

with collection.batch.dynamic() as batch:
    for idx, row in chunks_df.iterrows():
        # Construimos el diccionario de propiedades
        properties = {
            "text": row["text"],
            "doc_id": int(row["doc_id"]),
            "chunk_id": int(row["chunk_id"])
        }
        # Extraemos el vector numérico correspondiente
        vector = embeddings[idx].tolist()

        # Agregamos el objeto al lote
        batch.add_object(
            properties=properties,
            vector=vector
        )

# Verificar si hubo algún error en la inserción en lote
if collection.batch.failed_objects:
    print(f"Alerta: Fallaron {len(collection.batch.failed_objects)} objetos.")
else:
    print(f"¡Éxito! Se indexaron {len(chunks_df)} objetos y sus embeddings en Weaviate.")

Insertando datos en Weaviate...
¡Éxito! Se indexaron 79104 objetos y sus embeddings en Weaviate.


In [35]:
#Funcion de bsuqueda semantia
def weaviate_search(query_embedding, k=5):
    # Aplanar el vector a lista si viene en formato (1, D)
    if hasattr(query_embedding, "ndim") and query_embedding.ndim > 1:
        vector_busqueda = query_embedding.flatten().tolist()
    else:
        vector_busqueda = list(query_embedding)

    collection = client.collections.get(COLLECTION_WEAVIATE)

    # Ejecutar la consulta vectorial pura
    response = collection.query.near_vector(
        near_vector=vector_busqueda,
        limit=k,
        return_properties=["text", "doc_id", "chunk_id"],
        return_metadata=wvc.query.MetadataQuery(distance=True) # Traer el score de distancia
    )


    formatted_results = []
    for obj in response.objects:
        props = obj.properties
        metadata = {
            "doc_id": props.get("doc_id"),
            "chunk_id": props.get("chunk_id")
        }

        score = obj.metadata.distance if obj.metadata.distance is not None else 0.0

        formatted_results.append((obj.uuid, score, props.get("text"), metadata))

    return formatted_results

In [36]:
# Ejemplo de consulta con k=5
k_ejemplo = 5
resultados_weaviate = weaviate_search(query_vec, k=k_ejemplo)

print(f"--- TOP {k_ejemplo} RESULTADOS EN WEAVIATE ---")
for res in resultados_weaviate:
    uuid_val, score, text, meta = res
    print(f"UUID: {uuid_val} | Distancia: {score:.4f}")
    print(f"Metadata (Campos): {meta}")
    print(f"Texto extraído: {text[:140]}...")
    print("-" * 60)


--- TOP 5 RESULTADOS EN WEAVIATE ---
UUID: 1a3f077c-39ff-4bab-b4e9-c1accc980f3d | Distancia: 0.1297
Metadata (Campos): {'doc_id': 1391, 'chunk_id': 0}
Texto extraído: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device fo...
------------------------------------------------------------
UUID: dc8d1c59-b21a-42db-8277-b554aa193c10 | Distancia: 0.1382
Metadata (Campos): {'doc_id': 1, 'chunk_id': 0}
Texto extraído: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually...
------------------------------------------------------------
UUID: 88e2c303-ed65-4cb0-a176-b3b15293fc68 | Distancia: 0.1599
Metadata (Campos): {'doc_id': 1391, 'chunk_id': 1}
Texto extraído: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common princ...
----------------------

## ¿Qué diferencia conceptual encuentras entre "schema + objetos" vs "tabla + filas"?

**Respuesta:**

En el modelo de tabla + filas (SQL clásico) cada fila es un registro plano con columnas de tipos fijos, y las relaciones se arman con llaves foráneas y JOINs.

En Weaviate el schema define clases de objetos, y cada objeto encapsula sus propiedades junto con su vector como una sola entidad. Se identifica por UUID en vez de un índice numérico, y puede tener referencias directas a otros objetos (más estilo grafo que tabla). En la práctica se siente más como modelar "cosas con significado" que registros: el vector es parte del objeto, no una columna más.


## ¿Cómo describirías el trade-off de complejidad vs expresividad?

**Respuesta:**

Weaviate pide más configuración inicial que algo como FAISS o Chroma: hay que definir la colección, las propiedades con sus tipos, y acostumbrarse a trabajar con UUIDs. Para un prototipo simple se siente como trabajo extra.

A cambio, el motor hace mucho más por ti: búsqueda vectorial nativa, búsqueda híbrida (vectores + BM25), filtros sobre las propiedades y referencias entre objetos. En un proyecto real eso significa menos lógica que programar por fuera — cosas que con FAISS tendría que resolver yo con diccionarios y pandas, aquí las resuelve la base de datos. O sea: más complejidad al inicio, menos código después.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [37]:
### 1. Instalación e Inicialización
!pip install -q chromadb

import chromadb
from chromadb.config import Settings

# Inicializar cliente persistente en una carpeta local
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Crear o recuperar la colección
# Usamos la métrica 'cosine' por defecto para E5
collection_chroma = chroma_client.get_or_create_collection(name="wikipedia_chroma", metadata={"hnsw:space": "cosine"})

print("Cliente de ChromaDB listo.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [38]:
### 2. Inserción de datos
from tqdm.auto import tqdm
import gc

# Limpiar colección si hubo una interrupción previa para evitar duplicados
try:
    chroma_client.delete_collection(name="wikipedia_chroma")
except Exception:
    pass
collection_chroma = chroma_client.create_collection(
    name="wikipedia_chroma",
    metadata={"hnsw:space": "cosine"}
)

step = 2000
N = len(chunks_df)
ids_all = [str(i) for i in range(N)]

print("Iniciando inserción en ChromaDB...")
for i in tqdm(range(0, N, step)):
    j = min(i + step, N)
    collection_chroma.add(
        ids=ids_all[i:j],
        embeddings=embeddings[i:j].tolist(),
        documents=chunks_df["text"].iloc[i:j].tolist(),
        metadatas=chunks_df[["doc_id", "chunk_id"]].iloc[i:j].to_dict("records")
    )

gc.collect()
print(f"\nÉxito: Se han indexado {collection_chroma.count()} documentos.")

Iniciando inserción en ChromaDB...


  0%|          | 0/40 [00:00<?, ?it/s]


Éxito: Se han indexado 79104 documentos.


In [39]:
### 3. Función de búsqueda y ejemplo
def chroma_search(query_vec, k=5):
    # Chroma acepta una lista de vectores de consulta
    results = collection_chroma.query(
        query_embeddings=[query_vec.flatten().tolist()],
        n_results=k
    )

    # Formatear la salida para que coincida con el estándar del notebook
    formatted_results = []
    for i in range(len(results['ids'][0])):
        formatted_results.append({
            "id": results['ids'][0][i],
            "score": results['distances'][0][i],
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })
    return formatted_results

# Ejemplo de consulta
print(f"Buscando en Chroma: '{query_text}'\n")
resultados_ch = chroma_search(query_vec, k=5)

for i, res in enumerate(resultados_ch):
    print(f"Resultado {i+1} (Distancia Coseno: {res['score']:.4f}):")
    print(f"Texto: {res['text'][:150]}...")
    print("-" * 20)

Buscando en Chroma: 'Battery measuring'

Resultado 1 (Distancia Coseno: 0.1297):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------
Resultado 2 (Distancia Coseno: 0.1382):
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------
Resultado 3 (Distancia Coseno: 0.1599):
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------
Resultado 4 (Distancia Coseno: 0.1609):
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
--------------------
Resultado 5 (Distancia Coseno: 0.1614):
Texto: is achieved. Accepted av

## ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?

**Respuesta:**

Mucho más fácil. En Chroma no hay que definir esquemas ni tipos de campos: `create_collection` y `add` con ids, embeddings, documentos y metadatas, y listo. En Milvus tuve que declarar cada campo con su tipo y crear el índice aparte, y en Qdrant armar los PointStruct con payload. Aquí todo el pipeline salió en unas 20 líneas. Se nota que está pensado para prototipar rápido.


## ¿Qué limitaciones ves para un sistema en producción?

**Respuesta:**

Chroma corre embebido en el mismo proceso de Python y guarda todo en una carpeta local, así que no hay forma nativa de escalar horizontalmente ni de que varios servicios consulten la misma base de forma concurrente. Tampoco tiene el control fino de índices de Milvus, ni los filtros y la infraestructura distribuida de Qdrant o Weaviate. Con millones de vectores el índice HNSW en memoria se vuelve un problema. Para un prototipo o una app pequeña sobra; para producción con alto tráfico haría falta migrar a una base dedicada con cliente-servidor.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [40]:
%%bash
# Instalar PostgreSQL (Colab = Ubuntu 22.04 -> PostgreSQL 14)
apt-get -qq update
apt-get -qq -y install postgresql postgresql-contrib postgresql-server-dev-14 > /dev/null

# Compilar e instalar la extensión pgvector
cd /tmp
rm -rf pgvector
git clone --quiet --branch v0.7.4 https://github.com/pgvector/pgvector.git
cd pgvector
make -s > /dev/null && make -s install > /dev/null

# Arrancar el servidor y crear la base
service postgresql start
sudo -u postgres psql -q -c "ALTER USER postgres PASSWORD 'postgres';"
sudo -u postgres psql -q -c "CREATE DATABASE vectordb;" || true
echo "PostgreSQL + pgvector listos"

 * Starting PostgreSQL 14 database server
   ...done.
PostgreSQL + pgvector listos


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Note: switching to '103ac50f1a90b47a72003e8e8628a55ec372f202'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [41]:
#Conexion, extesion y creacion de tabla
!pip install -q psycopg2-binary pgvector

import psycopg2
import numpy as np
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host="localhost", dbname="vectordb",
    user="postgres", password="postgres"
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

register_vector(conn)  # permite pasar arrays de numpy directo en las queries

D = embeddings.shape[1]
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS documents;")
    cur.execute(f"""
        CREATE TABLE documents (
            id        INTEGER PRIMARY KEY,
            doc_id    INTEGER NOT NULL,
            chunk_id  INTEGER NOT NULL,
            text      TEXT NOT NULL,
            embedding vector({D}) NOT NULL
        );
    """)
print(f"Tabla 'documents' creada con vector({D}).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 63.1 MB/s eta 0:00:00
Tabla 'documents' creada con vector(768).


In [42]:
#INserccion
import io
from tqdm.auto import tqdm

def vec_to_pg(v):
    return "[" + ",".join(f"{x:.6f}" for x in v) + "]"

def clean(t):
    # COPY usa tab y salto de línea como separadores: hay que escaparlos
    return t.replace("\\", "\\\\").replace("\t", " ").replace("\n", " ").replace("\r", " ")

BATCH = 2000
texts = chunks_df["text"].tolist()
doc_ids = chunks_df["doc_id"].to_numpy()
chunk_ids = chunks_df["chunk_id"].to_numpy()

with conn.cursor() as cur:
    for start in tqdm(range(0, len(chunks_df), BATCH)):
        end = min(start + BATCH, len(chunks_df))
        buf = io.StringIO()
        for i in range(start, end):
            buf.write(f"{i}\t{int(doc_ids[i])}\t{int(chunk_ids[i])}\t"
                      f"{clean(texts[i])}\t{vec_to_pg(embeddings[i])}\n")
        buf.seek(0)
        cur.copy_expert(
            "COPY documents (id, doc_id, chunk_id, text, embedding) FROM STDIN",
            buf
        )

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM documents;")
    print(f"Filas insertadas: {cur.fetchone()[0]}")

  0%|          | 0/40 [00:00<?, ?it/s]

Filas insertadas: 79104


In [43]:
# Función pgvector_search y consulta de ejemplo
def pgvector_search(query_embedding, k=5):
    """
    Ejecuta SQL de similitud y retorna [(id, score, text, metadata), ...].
    El operador <=> de pgvector es DISTANCIA coseno (menor = mejor).
    """
    q = np.asarray(query_embedding, dtype=np.float32).flatten()
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT id, doc_id, chunk_id, text, embedding <=> %s AS distance
            FROM documents
            ORDER BY distance
            LIMIT %s;
            """,
            (q, k),
        )
        rows = cur.fetchall()

    return [
        (r[0], float(r[4]), r[3], {"doc_id": r[1], "chunk_id": r[2]})
        for r in rows
    ]

# Ejemplo con k=5
resultados_pg = pgvector_search(query_vec, k=5)

print(f"--- TOP 5 RESULTADOS EN PGVECTOR ---")
for idx, dist, text, meta in resultados_pg:
    print(f"ID: {idx} | Distancia coseno: {dist:.4f}")
    print(f"Metadata: {meta}")
    print(f"Texto: {text[:150]}...")
    print("-" * 50)

--- TOP 5 RESULTADOS EN PGVECTOR ---
ID: 10176 | Distancia coseno: 0.1297
Metadata: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------------------------------------
ID: 1 | Distancia coseno: 0.1382
Metadata: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------------------------------------
ID: 10177 | Distancia coseno: 0.1599
Metadata: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------------------------------------
ID: 37406 | Distancia coseno: 0.1609
Metadata: {'doc_id': 5067, 'chunk_id': 1}
Texto: ils. One wa

## ¿Qué tan "explicable" te parece esta aproximación vs las otras?

**Respuesta:**

Es la más explicable de todas. La búsqueda es una consulta SQL que se lee directamente: qué tabla, qué operador de distancia (`<=>` es distancia coseno), cómo ordena y cuántos resultados trae. Además puedo correr `EXPLAIN ANALYZE` y ver exactamente qué plan ejecutó la base. En Qdrant, Milvus o Weaviate todo eso queda escondido detrás de la API del cliente.

---

## ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?

**Respuesta:**

Los vectores viven junto al resto de los datos, así que puedo combinar la búsqueda por similitud con todo lo que ya sabe hacer SQL: JOIN contra otras tablas (usuarios, categorías, permisos), WHERE arbitrarios sobre la metadata, GROUP BY para agregar resultados, transacciones y backups. La búsqueda vectorial se vuelve una cláusula más dentro de una consulta normal, sin tener que sincronizar dos sistemas separados.

## ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?

**Respuesta:**

PostgreSQL es de un solo nodo: no tiene sharding nativo de vectores ni escalado horizontal como Milvus o Weaviate. Los índices ANN (IVFFlat/HNSW) compiten por RAM y CPU con el resto de la base, y con cientos de millones de vectores la latencia y el mantenimiento del índice se degradan. Para volúmenes pequeños o medianos (unos pocos millones de vectores) pgvector sobra y simplifica mucho la arquitectura; a gran escala las bases dedicadas ganan por cuantización, almacenamiento optimizado y distribución.